In [3]:
import urllib.request
import zipfile
from functools import partial
import os

chinook_url = 'http://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip'
if not os.path.exists('chinook.zip'):
    print('downloading chinook.zip ', end='')
    with urllib.request.urlopen(chinook_url) as response:
        with open('chinook.zip', 'wb') as f:
            for data in iter(partial(response.read, 4*1024), b''):
                print('.', end='', flush=True)
                f.write(data)

zipfile.ZipFile('chinook.zip').extractall()
assert os.path.exists('chinook.db')

In [4]:
from IPython.display import display
import pandas as pd

def sql(query):
    print()
    print(query)
    print()

def get_results(query):
    global engine
    q = query.statement if isinstance(query, sqlalchemy.orm.query.Query) else query
    return pd.read_sql(q, engine)

def display_results(query):
    df = get_results(query)
    display(df)
    sql(query)


In [7]:
# Exercise 1

import sqlalchemy
from sqlalchemy import create_engine

# 1. Create an engine object
# 'sqlite:///chinook.db' tells SQLAlchemy to use the SQLite driver
# and look for a file named chinook.db in the current directory
engine = create_engine('sqlite:///chinook.db')

# 2. Call the connect() method to obtain a connection
cur = engine.connect()

# --- Provided reflection code ---

# Extract classes from the chinook database
metadata = sqlalchemy.MetaData()
metadata.reflect(engine) # Reflect the tables

from sqlalchemy.ext.automap import automap_base

# Produce a set of mappings from this MetaData
Base = automap_base(metadata=metadata)

# Calling prepare() sets up mapped classes and relationships
Base.prepare()

# Map the classes (e.g., Base.classes.artists)
Artist = Base.classes.artists
Album = Base.classes.albums
Track = Base.classes.tracks

# Prepare an ORM session
from sqlalchemy.orm import sessionmaker
Session = sessionmaker(bind=engine)
session = Session()

print("Database engine created and session initialized successfully.")

Database engine created and session initialized successfully.


In [8]:
# Exercise 2

from sqlalchemy import inspect

# Use the inspect tool to get table names from the engine
inspector = inspect(engine)
table_names = inspector.get_table_names()

print("Tables in the Chinook Database:")
print("-" * 30)
for table in table_names:
    print(f"- {table}")

Tables in the Chinook Database:
------------------------------
- albums
- artists
- customers
- employees
- genres
- invoice_items
- invoices
- media_types
- playlist_track
- playlists
- tracks


In [9]:
# Exercise 3

# 1. Access the Track class from the automapped classes
Track = Base.classes.tracks

# 2. Create a query to get the first 3 tracks
# We use .limit(3) to restrict the results
three_tracks_query = session.query(Track).limit(3)

# 3. Use the provided display_results function to show the data
display_results(three_tracks_query)

for track in three_tracks_query:
    print(f"Track ID: {track.TrackId} | Name: {track.Name}")

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,None,342562,5510424,0.99
2,3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619,3990994,0.99



SELECT tracks."TrackId" AS "tracks_TrackId", tracks."Name" AS "tracks_Name", tracks."AlbumId" AS "tracks_AlbumId", tracks."MediaTypeId" AS "tracks_MediaTypeId", tracks."GenreId" AS "tracks_GenreId", tracks."Composer" AS "tracks_Composer", tracks."Milliseconds" AS "tracks_Milliseconds", tracks."Bytes" AS "tracks_Bytes", tracks."UnitPrice" AS "tracks_UnitPrice" 
FROM tracks
 LIMIT ? OFFSET ?

Track ID: 1 | Name: For Those About To Rock (We Salute You)
Track ID: 2 | Name: Balls to the Wall
Track ID: 3 | Name: Fast As a Shark


In [10]:
# Exercise 4

# 1. Map the classes for easy access
Track = Base.classes.tracks
Album = Base.classes.albums

# 2. Build the query joining Track and Album
# We select Track.Name and Album.Title specifically
query = session.query(Track.Name, Album.Title).\
        join(Album, Track.AlbumId == Album.AlbumId).\
        limit(20)

# 3. Display the results using your helper function
display_results(query)

,Name,Title
0,For Those About To Rock (We Salute You),For Those About To Rock We Salute You
1,Put The Finger On You,For Those About To Rock We Salute You
2,Let's Get It Up,For Those About To Rock We Salute You
3,Inject The Venom,For Those About To Rock We Salute You
4,Snowballed,For Those About To Rock We Salute You
5,Evil Walks,For Those About To Rock We Salute You
6,C.O.D.,For Those About To Rock We Salute You
7,Breaking The Rules,For Those About To Rock We Salute You
8,Night Of The Long Knives,For Those About To Rock We Salute You
9,Spellbound,For Those About To Rock We Salute You



SELECT tracks."Name" AS "tracks_Name", albums."Title" AS "albums_Title" 
FROM tracks JOIN albums ON tracks."AlbumId" = albums."AlbumId"
 LIMIT ? OFFSET ?



In [11]:
# Exercise 5

# 1. Map the classes
InvoiceItem = Base.classes.invoice_items
Track = Base.classes.tracks

# 2. Build the query
# We select the track name from the Track table
# and the quantity from the InvoiceItem table
query = session.query(Track.Name, InvoiceItem.Quantity).\
        join(InvoiceItem, Track.TrackId == InvoiceItem.TrackId).\
        limit(10)

# 3. Display the results
display_results(query)

,Name,Quantity
0,Balls to the Wall,1
1,Restless and Wild,1
2,Put The Finger On You,1
3,Inject The Venom,1
4,Evil Walks,1
5,Breaking The Rules,1
6,Dog Eat Dog,1
7,Overdose,1
8,Love In An Elevator,1
9,Janie's Got A Gun,1



SELECT tracks."Name" AS "tracks_Name", invoice_items."Quantity" AS "invoice_items_Quantity" 
FROM tracks JOIN invoice_items ON tracks."TrackId" = invoice_items."TrackId"
 LIMIT ? OFFSET ?



In [12]:
# Exercise 6

from sqlalchemy import func

# 1. Map the classes
Track = Base.classes.tracks
InvoiceItem = Base.classes.invoice_items

# 2. Build the aggregate query
# We select the Track Name and the SUM of the Quantity column
# We label the sum 'total_sold' so we can refer to it in our sorting
query = session.query(
            Track.Name,
            func.sum(InvoiceItem.Quantity).label('total_sold')
        ).\
        join(InvoiceItem, Track.TrackId == InvoiceItem.TrackId).\
        group_by(Track.Name).\
        order_by(func.sum(InvoiceItem.Quantity).desc()).\
        limit(10)

# 3. Display the results
display_results(query)

,Name,total_sold
0,The Trooper,5
1,Untitled,4
2,The Number Of The Beast,4
3,Sure Know Something,4
4,Hallowed Be Thy Name,4
5,Eruption,4
6,Where Eagles Dare,3
7,Welcome Home (Sanitarium),3
8,Sweetest Thing,3
9,Surrender,3



SELECT tracks."Name" AS "tracks_Name", sum(invoice_items."Quantity") AS total_sold 
FROM tracks JOIN invoice_items ON tracks."TrackId" = invoice_items."TrackId" GROUP BY tracks."Name" ORDER BY sum(invoice_items."Quantity") DESC
 LIMIT ? OFFSET ?



In [13]:
# Exercise 7

from sqlalchemy import func

# 1. Map all necessary classes
Artist = Base.classes.artists
Album = Base.classes.albums
Track = Base.classes.tracks
InvoiceItem = Base.classes.invoice_items

# 2. Build the multi-join aggregate query
query = session.query(
            Artist.Name,
            func.sum(InvoiceItem.Quantity).label('total_sold')
        ).\
        join(Album, Artist.ArtistId == Album.ArtistId).\
        join(Track, Album.AlbumId == Track.AlbumId).\
        join(InvoiceItem, Track.TrackId == InvoiceItem.TrackId).\
        group_by(Artist.Name).\
        order_by(func.sum(InvoiceItem.Quantity).desc()).\
        limit(10)

# 3. Display the results
display_results(query)

,Name,total_sold
0,Iron Maiden,140
1,U2,107
2,Metallica,91
3,Led Zeppelin,87
4,Os Paralamas Do Sucesso,45
5,Deep Purple,44
6,Faith No More,42
7,Lost,41
8,Eric Clapton,40
9,R.E.M.,39



SELECT artists."Name" AS "artists_Name", sum(invoice_items."Quantity") AS total_sold 
FROM artists JOIN albums ON artists."ArtistId" = albums."ArtistId" JOIN tracks ON albums."AlbumId" = tracks."AlbumId" JOIN invoice_items ON tracks."TrackId" = invoice_items."TrackId" GROUP BY artists."Name" ORDER BY sum(invoice_items."Quantity") DESC
 LIMIT ? OFFSET ?

